# Source-Grounded Business Summaries · real-time and batch example

**UNRELEASED candidate.** This notebook is prepared for the certified product but cannot run AWS operations until Software Sushi pins the real, release-approved Marketplace package ARN and image digest below. The product is not available for subscription yet. No model output is fabricated or saved in this notebook.

After release, subscribe to the AWS Marketplace product and use your own AWS account, SageMaker execution role, and private S3 bucket in `us-east-1`. Install `boto3` locally. The execution role must trust SageMaker and allow the purchased package plus the private input/output prefix. Your caller needs permission to create, describe, invoke and delete these demo resources and pass that role. Request capacity for one `ml.g6e.2xlarge` endpoint and, optionally, one Batch Transform instance.

**AWS infrastructure charges are separate from software charges and any software trial.** This example creates one GPU endpoint until you delete it; batch compute also incurs charges while running. Execute the cleanup cells, even after a failure. All example text is fictional. Review every model result: literal source evidence does not establish semantic support.

In [ ]:
import json
import os
import time
import uuid

import boto3
from botocore.exceptions import ClientError, WaiterError

# Release-owned constants: populated only from completed certification evidence.
APPROVED_CERTIFIED_MODEL_PACKAGE_ARN = ""
APPROVED_IMAGE_DIGEST = ""
REGION = "us-east-1"

configured_arn = os.environ.get("GROUNDED_SUMMARY_MODEL_PACKAGE_ARN", "").strip()
if not APPROVED_CERTIFIED_MODEL_PACKAGE_ARN or not APPROVED_IMAGE_DIGEST:
    raise RuntimeError("Unreleased candidate: no certified package/image is approved yet.")
if configured_arn != APPROVED_CERTIFIED_MODEL_PACKAGE_ARN:
    raise RuntimeError("Configure the exact release-approved Marketplace package ARN.")

session = boto3.Session(region_name=REGION)
sm = session.client("sagemaker")
runtime = session.client("sagemaker-runtime")
s3 = session.client("s3")
account_id = session.client("sts").get_caller_identity()["Account"]
role_arn = os.environ.get("SAGEMAKER_EXECUTION_ROLE_ARN", "").strip()
bucket = os.environ.get("SAGEMAKER_DEMO_BUCKET", "").strip()
if not role_arn.startswith(f"arn:aws:iam::{account_id}:role/") or not bucket:
    raise RuntimeError("Configure your account's execution role and private S3 bucket.")
s3.head_bucket(Bucket=bucket, ExpectedBucketOwner=account_id)
package = sm.describe_model_package(ModelPackageName=configured_arn)
if package.get("ModelPackageStatus") != "Completed" or not package.get("CertifyForMarketplace"):
    raise RuntimeError("The package is not completed and certified for Marketplace.")
containers = package.get("InferenceSpecification", {}).get("Containers", [])
if len(containers) != 1:
    raise RuntimeError("Expected the single qualified container.")
image_digest = containers[0].get("ImageDigest") or containers[0].get("Image", "").partition("@")[2]
if image_digest != APPROVED_IMAGE_DIGEST:
    raise RuntimeError("Package image differs from the release-approved digest.")

run_name = "ss-summary-demo-" + time.strftime("%Y%m%d-%H%M%S", time.gmtime()) + "-" + uuid.uuid4().hex[:6]
prefix = f"grounded-summary-demo/{run_name}"
tags = [{"Key": "SoftwareSushiDemoRun", "Value": run_name}]
model_created = False
config_created = False
endpoint_created = False
transform_created = False

## Bounded multilingual requests

Use one structured request per invocation, or one request per line in a line-split batch. Limits: 1–64 unique segments; 2,000 characters per segment; 16,000 combined ID-plus-text characters; 96 KiB body; 3,072 actual input tokens; 1,024 output tokens. Oversized inputs are rejected without truncation. Owner is an explicitly named person only, and deadline is an explicit valid ISO date only; otherwise null. This service accepts text, not audio, images, or PDFs.

In [ ]:
examples = {
    "en": "Maya Chen will send the signed transition plan by 2027-06-18.",
    "es": "Lucía Mena enviará el cronograma firmado antes del 2027-06-18.",
    "pt": "João Luís enviará o cronograma assinado até 2027-06-18.",
}
requests = [
    {"language": language, "segments": [{"id": "s1", "text": text}]}
    for language, text in examples.items()
]

## Create one temporary real-time endpoint

Run this cell only when ready to test. Wait for `InService`, then invoke immediately and delete the endpoint using the next cleanup cell. If creation or waiting fails, run cleanup before retrying. Keep this notebook open until cleanup finishes.

In [ ]:
sm.create_model(
    ModelName=run_name,
    ExecutionRoleArn=role_arn,
    PrimaryContainer={"ModelPackageName": configured_arn},
    EnableNetworkIsolation=True,
    Tags=tags,
)
model_created = True
sm.create_endpoint_config(
    EndpointConfigName=run_name,
    ProductionVariants=[{
        "VariantName": "AllTraffic", "ModelName": run_name,
        "InstanceType": "ml.g6e.2xlarge", "InitialInstanceCount": 1,
        "InitialVariantWeight": 1,
        "ContainerStartupHealthCheckTimeoutInSeconds": 600,
        "ModelDataDownloadTimeoutInSeconds": 1200,
    }],
    Tags=tags,
)
config_created = True
sm.create_endpoint(EndpointName=run_name, EndpointConfigName=run_name, Tags=tags)
endpoint_created = True
sm.get_waiter("endpoint_in_service").wait(
    EndpointName=run_name, WaiterConfig={"Delay": 30, "MaxAttempts": 40},
)

In [ ]:
for request in requests:
    response = runtime.invoke_endpoint(
        EndpointName=run_name, ContentType="application/json", Accept="application/json",
        Body=json.dumps(request, ensure_ascii=False).encode("utf-8"),
    )
    try:
        result = json.loads(response["Body"].read())
    finally:
        response["Body"].close()
    assert result["model"] == "grounded-summary" and result["version"] == "0.1.0"
    assert result["review_required"] is True
    source = {s["id"]: s["text"] for s in request["segments"]}
    for item in result["summary"] + result["actions"]:
        evidence = item["evidence"]
        text = source[evidence["segment_id"]]
        assert evidence["quote"] == text and evidence["start"] == 0 and evidence["end"] == len(text)
    print(json.dumps(result, ensure_ascii=False, indent=2))
# Exact citation location is checked above; semantic support still requires review.

In [ ]:
# Stop endpoint charges before the optional batch example. Retain the model for batch.
if endpoint_created:
    sm.delete_endpoint(EndpointName=run_name)
    sm.get_waiter("endpoint_deleted").wait(
        EndpointName=run_name, WaiterConfig={"Delay": 15, "MaxAttempts": 80},
    )
    endpoint_created = False
if config_created:
    sm.delete_endpoint_config(EndpointConfigName=run_name)
    config_created = False

## Optional line-split Batch Transform

This cell needs the separate GPU batch quota. It uploads three fictional requests to your private bucket using server-side encryption. Batch compute stops when the job completes. Input/output objects are retained in your bucket for inspection; remove only this run's prefix later if appropriate.

In [ ]:
input_key = f"{prefix}/input/examples.jsonl"
s3.put_object(
    Bucket=bucket, Key=input_key, ExpectedBucketOwner=account_id,
    Body=("\n".join(json.dumps(r, ensure_ascii=False) for r in requests) + "\n").encode("utf-8"),
    ContentType="application/jsonlines", ServerSideEncryption="AES256",
)
sm.create_transform_job(
    TransformJobName=run_name, ModelName=run_name, BatchStrategy="SingleRecord",
    MaxConcurrentTransforms=1, MaxPayloadInMB=1,
    TransformInput={
        "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{bucket}/{input_key}"}},
        "ContentType": "application/jsonlines", "SplitType": "Line",
    },
    TransformOutput={
        "S3OutputPath": f"s3://{bucket}/{prefix}/output/",
        "Accept": "application/jsonlines", "AssembleWith": "Line",
    },
    TransformResources={"InstanceType": "ml.g6e.2xlarge", "InstanceCount": 1}, Tags=tags,
)
transform_created = True
sm.get_waiter("transform_job_completed_or_stopped").wait(
    TransformJobName=run_name, WaiterConfig={"Delay": 30, "MaxAttempts": 120},
)
job = sm.describe_transform_job(TransformJobName=run_name)
if job["TransformJobStatus"] != "Completed":
    raise RuntimeError("Batch did not complete; inspect the job and run cleanup.")
output = s3.get_object(
    Bucket=bucket, Key=f"{prefix}/output/examples.jsonl.out", ExpectedBucketOwner=account_id,
)
try:
    for line in output["Body"].read().decode("utf-8").splitlines():
        print(json.dumps(json.loads(line), ensure_ascii=False, indent=2))
finally:
    output["Body"].close()

## Cleanup after success or failure

Run this cell even if an earlier operation failed. It acts only on this unique demo run. It stops an unfinished batch job, deletes any remaining endpoint/configuration, and removes the demo model after compute stops. It preserves S3 input/output evidence. Check the AWS console if a cleanup call fails; closing the notebook does not stop an endpoint.

In [ ]:
cleanup_errors = []
batch_active = False
if transform_created:
    try:
        job = sm.describe_transform_job(TransformJobName=run_name)
        status = job["TransformJobStatus"]
        batch_active = status in ("InProgress", "Stopping")
        if status == "InProgress":
            sm.stop_transform_job(TransformJobName=run_name)
        if batch_active:
            sm.get_waiter("transform_job_completed_or_stopped").wait(
                TransformJobName=run_name, WaiterConfig={"Delay": 15, "MaxAttempts": 80},
            )
            batch_active = False
    except (ClientError, WaiterError) as exc:
        cleanup_errors.append(exc)
# Endpoint cleanup still runs if a batch operation failed.
if endpoint_created:
    try:
        state = sm.describe_endpoint(EndpointName=run_name)["EndpointStatus"]
        if state != "Deleting":
            sm.delete_endpoint(EndpointName=run_name)
        sm.get_waiter("endpoint_deleted").wait(
            EndpointName=run_name, WaiterConfig={"Delay": 15, "MaxAttempts": 80},
        )
        endpoint_created = False
    except (ClientError, WaiterError) as exc:
        cleanup_errors.append(exc)
if config_created and not endpoint_created:
    try:
        sm.delete_endpoint_config(EndpointConfigName=run_name)
        config_created = False
    except ClientError as exc:
        cleanup_errors.append(exc)
if model_created and not endpoint_created and not batch_active:
    try:
        sm.delete_model(ModelName=run_name)
        model_created = False
    except ClientError as exc:
        cleanup_errors.append(exc)
if cleanup_errors:
    raise RuntimeError("Some cleanup calls failed; inspect this demo run in AWS before closing.") from cleanup_errors[0]
print(f"Demo compute removed. Private evidence remains at s3://{bucket}/{prefix}/")
